In [ ]:
# ==========================================
# 1. standard library imports 
# ==========================================
import os
import glob
import shutil
import zipfile
import warnings
import calendar
import urllib.request
import requests

# ==========================================
# 2. data analysis math thingies imports
# ==========================================
import numpy as np
import pandas as pd
import xarray as xr

# ==========================================
# 3. plotting imports
# ==========================================
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# suppress annoying warnings 
warnings.filterwarnings('ignore')

In [ ]:
# ==========================================
# 2. Regional filtering and download+ QC flags and filtering by number of cycles 
# ==========================================
# Format: "Region_Name": (lat_min, lat_max, lon_min, lon_max)
TARGET_REGIONS = {
    "Arabian_Sea": (5, 20, 50, 100),       # Your custom Lon limit (>=50E up to 100E)
    "Bay_of_Bengal": (5, 20, 80, 100),
    "Equatorial_IO": (-5, 5, 50, 100)
}

BASE_URL = "https://data-argo.ifremer.fr/dac"

def download_bioargo_data(index_df):
    """
    Loops through target ocean regions, filters the Argo index for quality 
    profiles (>20 cycles containing CHLA), and downloads the NetCDF files.
    """
    print("starting download pipeline...\n")
    
    for region_name, bounds in TARGET_REGIONS.items():
        lat_min, lat_max, lon_min, lon_max = bounds
        
        print(f"processing region: {region_name.replace('_', ' ')}")
        print(f"bounds: lat({lat_min} to {lat_max}), lon({lon_min} to {lon_max})")

        # 1. spatial & parameter filtering
        spatial_mask = (index_df['latitude'] >= lat_min) & (index_df['latitude'] <= lat_max) & \
                       (index_df['longitude'] >= lon_min) & (index_df['longitude'] <= lon_max)
        chl_mask = index_df['parameters'].str.contains('CHLA', na=False)
        
        region_df = index_df[spatial_mask & chl_mask].copy()
        
        if region_df.empty:
            print(f"whoops! no floats found matching criteria in {region_name}. Skipping.\n")
            continue
            
        # extract the WMO ID from the file path string
        region_df['wmo'] = region_df['file'].str.split('/').str[1]

        # 2. cycle no filter: only keep floats with >20 profiles/cycles
        profile_counts = region_df.groupby('wmo').size()
        frequent_floats = profile_counts[profile_counts > 20].index.tolist()
        
        print(f"found {len(profile_counts)} total floats. filters applied: CHLA present & >20 cycles.")
        print(f"{len(frequent_floats)} floats passed the quality check.")

        # 3. setup local storage (ignored by Git automatically via our planned .gitignore)
        download_dir = f"fixed_{region_name}_chl_bioargo"
        os.makedirs(download_dir, exist_ok=True)
        
        # lookup mapping each WMO to its respective DAC (Data Assembly Center) so we dont get random files from the pacific or smn accidently 
        wmo_to_dac = dict(zip(region_df['wmo'], region_df['file'].str.split('/').str[0]))

        # 4. download loop
        print(f"downloading NetCDF files to '{download_dir}'...")
        for wmo in sorted(frequent_floats):
            file_name = f"{wmo}_Sprof.nc"
            save_path = os.path.join(download_dir, file_name)

            # check if file is already downloaded so you dont spend 30 mins and eat storage again 
            if os.path.exists(save_path):
                continue

            dac = wmo_to_dac.get(wmo, 'coriolis')
            url = f"{BASE_URL}/{dac}/{wmo}/{file_name}"

            try:
                response = requests.get(url, stream=True, timeout=15)
                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"   + saved {file_name}")
                else:
                    print(f"   - failed {wmo} (Status Code: {response.status_code})")
            except Exception as e:
                print(f" whoops error fetching float {wmo}: {e}")
                
        print(f"yay done processing for {region_name}.\n" + "-"*50 + "\n")

In [ ]:
def extract_float_coordinates(folder_name):
    """
    reads all NetCDF files in a folder, calculates the mean position 
    (centroid) for each float, and returns a DataFrame.
    """
    # look for files inside wtv folder is saved as
    file_list = glob.glob(os.path.join(folder_name, "*.nc"))
    
    if not file_list:
        print(f"??No NetCDF files found in '{folder_name}'.")
        return pd.DataFrame()

    data = []
    for file in file_list:
        try:
            with xr.open_dataset(file) as ds:
                # extract WMO from the filename safely across different operating systems
                wmo_id = os.path.basename(file).split('_')[0]
                
                data.append({
                    'wmo': wmo_id,
                    'latitude': ds['LATITUDE'].values.mean(),
                    'longitude': ds['LONGITUDE'].values.mean()
                })
        except Exception as e:
            print(f" whoops could not read {os.path.basename(file)}: {e}")

    return pd.DataFrame(data)

In [ ]:
def plot_float_positions(coords_df, region_name, bounds):
    """
    plots the float centroids on a cartopy map for verification so all floats are in the right place
    and saves the image locally.
    """
    if coords_df.empty:
        print(f"?? no data to plot for {region_name}")
        return

    lon_min, lon_max, lat_min, lat_max = bounds

    # setup the Plot
    fig = plt.figure(figsize=(10, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())

    # set dynamic map boundaries based on the region
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # add geography
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax.add_feature(cfeature.COASTLINE, linewidth=1.5, zorder=2)

    # plot the float centroids
    plt.scatter(coords_df['longitude'], coords_df['latitude'],
                color='darkblue', s=50, alpha=0.8, transform=ccrs.PlateCarree(),
                label=f'{region_name.replace("_", " ")} Floats', zorder=3, edgecolors='white')

    # gridlines so you can see coordinates 
    gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False

    # title and layout
    clean_title = region_name.replace("_", " ")
    plt.title(f'Argo Float Centroids in the {clean_title}', fontsize=14, pad=15)
    plt.legend(loc='lower left')
    
    # create a local directory for verification plots
    os.makedirs("plots", exist_ok=True)
    output_path = f"plots/{region_name}_centroids.png"
    
    # save locally to check your work. (Our .gitignore will hide this folder from GitHub!)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"yay verification plot saved locally to: {output_path}")

In [ ]:
# define the map limits for each region: (lon_min, lon_max, lat_min, lat_max)
PLOT_BOUNDS = {
    "Arabian_Sea": (50, 85, 5, 25),
    "Bay_of_Bengal": (80, 100, 5, 20),
    "Equatorial_IO": (50, 100, -5, 5)
}

# loop to run logic for all three regions
for region, map_limits in PLOT_BOUNDS.items():
    folder_path = f"fixed_{region}_chl_bioargo"
    
    print(f"\n extracting coordinates for {region}...")
    df_coords = extract_float_coordinates(folder_path)
    
    print(f" generating map for {region}...")
    plot_float_positions(df_coords, region, map_limits)

In [ ]:
def calculate_regional_climatology(base_dir, region_name):
    """
    loads regional NetCDF files, filters by quality control masks, 
    interpolates data onto a standard depth grid, calculates monthly 
    climatologies, and exports a CSV file.
    """
    print(f"starting processing pipeline for: {region_name}")
    
    # define standard vertical pressure (depth) grid 
    # steps: 0-200m every 5m, and 250m-2000m every 50m
    std_grid = np.concatenate([np.arange(0, 205, 5), np.arange(250, 2050, 50)])
    
    # initialize dictionary to collect profile lists for months Jan (1) to Dec (12)
    monthly_data = {month: [] for month in range(1, 13)}
    
    # get all .nc files inside the specified directory
    file_list = glob.glob(os.path.join(base_dir, "*.nc"))
    if not file_list:
        print(f"?? no NetCDF files found in {base_dir}. skipping region.")
        return

    for file_path in file_list:
        try:
            with xr.open_dataset(file_path, decode_times=True) as ds:
                # 1. extract post-calibration scientific adjusted variables
                chla = ds['CHLA_ADJUSTED'].values.copy()
                pres = ds['PRES_ADJUSTED'].values
                qc = ds['CHLA_ADJUSTED_QC'].values.astype(str)
                time = ds['JULD'].values

                # 2. quality control (QC) masking
                # keep data flagged as: 1 (Good), 2 (Probably Good), 5 (Value Changed), 8 (Estimated)
                valid_qc_mask = np.isin(qc, ['1', '2', '5', '8'])
                chla[~valid_qc_mask] = np.nan
                chla[chla <= 0] = np.nan  # drop unphysical zero or negative values and dont treat as zero or they mess w climatology means

                # 3. iterate through individual profiles (N_PROF)
                for i in range(ds.dims['N_PROF']):
                    month = pd.to_datetime(time[i]).month
                    p_levels = pres[i, :]
                    c_levels = chla[i, :]

                    # only interpolate if we have enough physical data points
                    if np.sum(~np.isnan(c_levels)) > 3:
                        # map uneven sensor readings cleanly onto our standard vertical grid
                        interp_c = np.interp(std_grid, p_levels, c_levels, left=np.nan, right=np.nan)
                        monthly_data[month].append(interp_c)
                        
        except Exception as e:
            print(f"   skipping file {os.path.basename(file_path)} due to error: {e}")

    # 4. compute monthly means across our standard depth grid rows
    monthly_means = {}
    for month in range(1, 13):
        data_matrix = monthly_data[month]
        if data_matrix:
            monthly_means[month] = np.nanmean(data_matrix, axis=0)
        else:
            # if a month has zero data, fill it with NaNs so the DataFrame matches grid size
            monthly_means[month] = np.zeros(len(std_grid)) * np.nan

    # 5. build final DataFrame and export climatology dataset
    df_climatology = pd.DataFrame(monthly_means, index=std_grid)
    df_climatology.index.name = 'Depth_Pressure_dbar'
    
    output_filename = f"{region_name}_Climatology.csv"
    df_climatology.to_csv(output_filename)
    
    print(f"dones monthly climatology saved to: {output_filename}\n" + "="*50)

In [ ]:
# map regions directly to the clean folder structures we created before 
REGIONS_CONFIG = {
    "Arabian_Sea": "fixed_Arabian_Sea_chl_bioargo",
    "Bay_of_Bengal": "fixed_Bay_of_Bengal_chl_bioargo",
    "Equatorial_IO": "fixed_Equatorial_IO_chl_bioargo"
}

# run the calculation for all regions 
for region, directory in REGIONS_CONFIG.items():
    calculate_regional_climatology(directory, region)

In [ ]:
def calculate_sst_climatology(file_path):
    """
    loads the global NOAA OISST NetCDF file, slices it into the 3 target 
    ocean basins, computes the regional spatial averages, and returns 
    a single compiled DataFrame for all regions.
    """
    print("loading local NOAA OISST dataset...")
    if not os.path.exists(file_path):
        print(f"error: {file_path} not found. please download.")
        return pd.DataFrame()
        
    ds = xr.open_dataset(file_path)

    # define your specific regions 
    regions_config = {
        "Arabian_Sea": {"lat": (5, 25), "lon": (50, 75)},
        "Bay_of_Bengal": {"lat": (5, 22), "lon": (80, 100)},
        "Equatorial_IO": {"lat": (-5, 5), "lon": (60, 90)}
    }

    sst_results = {}

    print("Slicing and calculating monthly means across target basins...")
    for name, bounds in regions_config.items():
        lat_min, lat_max = bounds["lat"]
        lon_min, lon_max = bounds["lon"]

        # safety Check: handle descending latitudes (89.5 down to -89.5) gracefully
        if ds.lat.values[0] > ds.lat.values[-1]:
            lat_slice = slice(lat_max, lat_min)
        else:
            lat_slice = slice(lat_min, lat_max)

        # subset data to region
        ds_region = ds.sel(lat=lat_slice, lon=slice(lon_min, lon_max))

        # calculate space mean, then aggregate by month to build a long-term climatology
        spatial_mean = ds_region['sst'].mean(dim=['lat', 'lon'])
        monthly_clim = spatial_mean.groupby('time.month').mean('time').values

        sst_results[name] = monthly_clim

    # format output into a clean, presentation-ready dataset
    months_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    df_sst = pd.DataFrame(sst_results, index=months_labels)
    df_sst.index.name = 'Month'
    
    return df_sst

In [ ]:
def plot_final_climatology_grid(df_sst):
    """
    generates a 12x3 matrix subplot displaying vertical Chlorophyll profiles 
    from 0-300m and correlates it with NOAA SST averages for all regions.
    saves the final plot locally under the 'plots' directory.
    """
    print("constructing the 12x3 Chlorophyll-SST matrix plot...")
    
    # map directly to the CSV files created by your script
    regions_files = {
        "Arabian Sea": "Arabian_Sea_Climatology.csv",
        "Bay of Bengal": "Bay_of_Bengal_Climatology.csv",
        "Equatorial IO": "Equatorial_IO_Climatology.csv"
    }

    months_str = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    global_chl_max = 0.0
    loaded_dfs = {}

    # pre-load files and isolate maximum value within upper 300m limits for universal scaling
    for region_name, filename in regions_files.items():
        if os.path.exists(filename):
            df = pd.read_csv(filename, index_col=0)
            df.index = pd.to_numeric(df.index)
            loaded_dfs[region_name] = df

            upper_300m_df = df.loc[df.index <= 300]
            if not upper_300m_df.empty:
                max_val = upper_300m_df.max().max()
                if max_val > global_chl_max:
                    global_chl_max = max_val

    # lock global X scale with a padding margin
    x_max_limit = global_chl_max * 1.1 if global_chl_max > 0 else 2.0

    fig, axes = plt.subplots(12, 3, figsize=(18, 40), constrained_layout=True)

    for row_idx, month_name in enumerate(months_str):
        col_name = str(row_idx + 1)  # Months columns are labeled '1' through '12'

        for col_idx, (region_name, _) in enumerate(regions_files.items()):
            ax = axes[row_idx, col_idx]

            if region_name in loaded_dfs:
                df = loaded_dfs[region_name]

                if col_name in df.columns:
                    data = df[[col_name]].dropna()
                    x = data[col_name].values
                    y = data.index.values

                    ax.plot(x, y, color='darkgreen', linewidth=1.5)

                    if len(x) > 0:
                        max_idx = np.argmax(x)
                        dcm_depth = y[max_idx]
                        plot_depth = 5 if dcm_depth == 0 else dcm_depth
                        ax.axhline(y=plot_depth, color='darkgreen', linestyle='--', alpha=0.8, linewidth=1.5)

            # uniform axes formatting across the whole subplot matrix
            ax.set_ylim(300, 0) # Focus on upper 300 meters
            ax.set_xlim(0, x_max_limit)

            ax.xaxis.set_ticks_position('top')
            ax.xaxis.set_label_position('top')
            ax.set_xlabel('Chl-a (mg/m³)')
            ax.grid(True, linestyle=':', alpha=0.5)

            if col_idx == 0:
                ax.set_ylabel('Depth (m)')
                ax.text(-0.35, 0.5, month_name, transform=ax.transAxes,
                        fontsize=12, fontweight='bold', rotation=90, va='center')

            if region_name in df_sst.columns:
                sst_val = df_sst.loc[month_name, region_name]
                ax.text(0.95, 0.95, f'SST: {sst_val:.1f}°C',
                        transform=ax.transAxes, fontsize=10, fontweight='bold',
                        ha='right', va='top', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

            if row_idx == 0:
                ax.set_title(region_name, fontsize=14, fontweight='bold', pad=30)

    # save to 'plots/' folder
    os.makedirs("plots", exist_ok=True)
    output_img_path = "plots/Master_Climatology_Profiles.png"
    plt.savefig(output_img_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"profile matrix saved to: {output_img_path}")

In [ ]:
# =====================================================================
# 3. PIPELINE EXECUTION 
# =====================================================================
if __name__ == "__main__":
    
    # ---- STEP A: download and map argo floats ----
    print(" Fetching the global Argo bio-profile index...")
    index_url = "https://data-argo.ifremer.fr/argo_bio-profile_index.txt.gz"
    global_df = pd.read_csv(index_url, skiprows=8)
    
    download_bioargo_data(global_df)

    # ---- STEP B: extract coordinates & map locations ----
    for region, map_limits in PLOT_BOUNDS.items():
        folder_path = f"fixed_{region}_chl_bioargo"
        df_coords = extract_float_coordinates(folder_path)
        plot_float_positions(df_coords, region, map_limits)
    print(" local coordinate checks and maps generated inside 'plots/' folder.")

    # ---- STEP C: calculate profile climatology ----
    for region in TARGET_REGIONS.keys():
        calculate_regional_climatology(f"fixed_{region}_chl_bioargo", region)

    # ---- STEP D: calculate sea surface temperature climatology ----
    noaa_url = "https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2/sst.mnmean.nc"
    local_sst_file = "sst.mnmean.nc"
    if not os.path.exists(local_sst_file):
        print("local NOAA OISST file missing. starting download...")
        urllib.request.urlretrieve(noaa_url, local_sst_file)
    
    df_sst_final = calculate_sst_climatology(local_sst_file)
    df_sst_final.to_csv("regional_SST_climatology.csv")

    # ---- STEP E: generate the plots grid ----
    # re-naming columns of df_sst_final to match exact layout keys ("Arabian Sea" instead of "Arabian_Sea")
    df_sst_plotting = df_sst_final.rename(columns={
        "Arabian_Sea": "Arabian Sea",
        "Bay_of_Bengal": "Bay of Bengal",
        "Equatorial_IO": "Equatorial IO"
    })
    plot_final_climatology_grid(df_sst_plotting)
    
    print("\ndonedonedone")f